# 01 — Dataset exploration and leakage audit

## Purpose

Use this notebook **before training** to understand the validated manifest, inspect real/fake examples, and audit the partitions that support the dissertation claims. It must not become an alternative dataset implementation: reusable parsing, filtering, splitting, and validation belong under `src/datasets/`.

Expected inputs: a manifest and persisted split-assignment table. Expected outputs: documented observations and optional audit figures/tables—never model results. Run cells top-to-bottom after completing the referenced TODOs.

In [ ]:
# SETUP TODO 1: Import Path, pandas, plotting tools, PIL, and reusable dataset validators.
# SETUP TODO 2: Add the repository root to imports by installing the package or launching
#                 Jupyter from the repository; avoid fragile sys.path edits in final work.
# SETUP TODO 3: Define manifest/split paths from a resolved config rather than duplicating them.
# SETUP TODO 4: Set display options and a plotting style without altering source data.

# from pathlib import Path
# import pandas as pd
# import matplotlib.pyplot as plt
# CONFIG_PATH = Path('../configs/baseline.yaml')

## 1. Load the dataset manifest

Load metadata, not all image pixels. Confirm that the file identity (path, size, checksum/version) is recorded so later experiments use the same manifest. Loading through the project validator prevents this notebook from applying different label coercions than training.

In [ ]:
# TODO 1: Load the resolved YAML using src.utils.config.load_config.
# TODO 2: Load CSV/Parquet metadata using the same function used by AIDetectionDataset.
# TODO 3: Print row/column counts and a manifest hash; do not display sensitive paths.
# TODO 4: Assert there is one row per intended sample before continuing.

# manifest = ...
# display(manifest.head())

## 2. Inspect metadata and schema

Check required columns (`sample_id`, `image_path`, `label`, `generator`) plus provenance columns such as `source_group`, dataset source, prompt/original ID, and content hash. Missing provenance can make a random split appear valid while near-duplicates leak across partitions.

In [ ]:
# TODO 1: Display column names, dtypes, null counts, unique counts, and representative values.
# TODO 2: Check duplicate sample IDs, image paths, content hashes, and source groups.
# TODO 3: Verify every path resolves under the configured data root and every file exists.
# TODO 4: Pre-scan image decodability; record corrupt files rather than silently skipping them.
# TODO 5: Document any exclusion rule and regenerate/version the manifest rather than editing rows here.

# schema_summary = ...
# display(schema_summary)

## 3. Check labels and generators

Verify the invariant 0=real and 1=fake (or change it once, everywhere, before implementation). Real records should use the reserved generator name consistently. Unknown spelling variants must not become accidental extra generators.

In [ ]:
# TODO 1: List exact label values and reject anything outside {0, 1}.
# TODO 2: Cross-tabulate label by generator and investigate impossible combinations.
# TODO 3: List generator spellings, whitespace/case variants, and sample counts.
# TODO 4: Confirm every fake record has a known generator and real records follow the sentinel policy.
# TODO 5: Save corrections in the upstream manifest-building process, not ad hoc notebook variables.

# display(pd.crosstab(manifest['generator'], manifest['label'], margins=True))

## 4. View a stratified image sample

Manual inspection catches wrong paths, labels, colour modes, watermarks, borders, resolution artefacts, and obvious dataset-source confounds. Sample reproducibly across real images and every fake generator; do not select only attractive examples.

In [ ]:
# TODO 1: Draw a fixed-seed sample per label/generator and report selected sample IDs.
# TODO 2: Open with the same EXIF/RGB policy planned for DetectorDataset.
# TODO 3: Display image, label, generator, size, mode, format, and provenance group.
# TODO 4: Look for class/generator-specific watermarks, padding, compression, or resolutions.
# TODO 5: Record observations in markdown; never transform or overwrite raw files here.

# sampled_rows = ...
# fig, axes = ...

## 5. Class and generator balance

Overall balance can hide severe imbalance inside a generator or split. Examine counts and proportions, but do not automatically rebalance: sampling and loss weighting change the effective training distribution and must be configured deliberately.

In [ ]:
# TODO 1: Calculate counts/proportions by label, generator, dataset source, and important intersections.
# TODO 2: Plot counts with values/supports visible; avoid plots that conceal tiny groups.
# TODO 3: Compare image dimensions/formats/compression proxies across labels and generators.
# TODO 4: Decide whether imbalance is handled by sampling, loss weighting, or metrics—using training data only.

# balance_table = ...
# display(balance_table)

## 6. Inspect train/validation/test and unseen-generator partitions

This is the most important audit. Check sample IDs, paths, hashes, prompts/originals, and source groups for overlap. For leave-one-generator-out, generator D must be absent from all model-development partitions. Its adaptation pool and final test pool must be created first and remain disjoint.

In [ ]:
# TODO 1: Load persisted split assignments and join to the manifest by unique sample_id.
# TODO 2: Assert every manifest sample has exactly one legal assignment (or a documented exclusion).
# TODO 3: Assert pairwise disjoint sample IDs, paths, content hashes, and source groups.
# TODO 4: Tabulate label/generator/group counts for train, validation, test, adaptation_pool, unseen_test.
# TODO 5: Assert the held-out generator is absent from train/validation/threshold selection.
# TODO 6: Assert every fine-tuning subset ID is in adaptation_pool and never unseen_test.
# TODO 7: Record achieved rather than merely requested split fractions.

# split_audit = ...
# assert ...

## Implementation checklist

- [ ] Load through the same validated manifest/config path used by experiments.
- [ ] Verify labels, generator naming, file existence, and decodability.
- [ ] Inspect reproducible examples from every label/generator.
- [ ] Quantify class, generator, source, format, and resolution balance.
- [ ] Audit sample/group/hash overlap across all partitions.
- [ ] Confirm held-out generator and final test isolation.
- [ ] Save observations with manifest/split identity; do not generate model results here.